<img src="https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/logo-epfl-1024x576.png" width="140px" alt="EPFL_logo">

## Image Processing Laboratory Notebooks
---

This Jupyter notebook is part of a series of computer laboratories which are designed
to teach image-processing programming; they are running on the EPFL's Noto server. They are the practical complement of the theoretical lectures of the EPFL's Master course 
[**MICRO-511 Image Processing I**](https://moodle.epfl.ch/course/view.php?id=522) taught by Prof. M. Unser and Prof. D. Van de Ville.

The project is funded by the Center for Digital Education and the School of Engineering. It is owned by the [Biomedical Imaging Group](https://bigwww.epfl.ch/). 
The distribution or the reproduction of the notebook is strictly prohibited without the written consent of the authors.  © EPFL 2026.

**Authors**: 
    [Pol del Aguila Pla](mailto:pol.delaguilapla@epfl.ch), 
    [Kay Lächler](mailto:kay.lachler@epfl.ch),
    [Alejandro Noguerón Arámburu](mailto:alejandro.nogueronaramburu@epfl.ch),
    [Yan Liu](mailto:yan.liu@epfl.ch), and
    [Daniel Sage](mailto:daniel.sage@epfl.ch).
    
---

To ensure your work is graded correctly by our automated system, **do not create new cells or delete/rearrange/copy existing ones** when you submit. The current cells contain hidden metadata required for the auto-grader to identify your solutions. If you create temporary cells for testing during your work, remember to clean them up before submission.

# Lab 2.1: Filtering
**Released**: Thursday November 5, 2026

**Submission**: Monday November 16, 2026 before 23:59 on [Moodle](https://moodle.epfl.ch/course/view.php?id=522)

**Grade weight**: Lab 2 (16 points), 10% of the overall grade

**Related lectures**: Chapter 3

Double-click on this cell, fill in your name and SCIPER number below, and run the next cell to verify your identity in Noto.

### Student Name:
### SCIPER:

In [ ]:
import getpass

# This line recovers your camipro number to mark the images with your ID
uid = (
    int(getpass.getuser().split("-")[2])
    if len(getpass.getuser().split("-")) > 2
    else ord(getpass.getuser()[0])
)
print(f"SCIPER: {uid}")

## Imports
Run the next cell to import the libraries and load the images that we will use throughout the lab. The images are loaded as `float64`, so that the results of the filters (which can be negative, or larger than $255$) are stored without clipping or rounding.

In [ ]:
# Configure plotting as dynamic
%matplotlib widget

# Import standard required packages for this exercise
import time
import matplotlib.pyplot as plt
import numpy as np
import skimage.io as io
import scipy.ndimage as ndi
import ipywidgets as widgets
from interactive_kit import imviewer as viewer

# Load images to be used in this exercise
bikesgray = io.imread("images/bikesgray.tif").astype(np.float64)
camera = io.imread("images/camera-16bits.tif").astype(np.float64)
spots = io.imread("images/spots.tif").astype(np.float64)

# Filtering (9 points)

In this lab you will review the basics of digital image filtering: how to implement a convolution (the main ingredient of filtering), separable and non-separable filters, filter design and boundary conditions. The goal is to fully understand the filtering algorithm at a pixel-by-pixel level by implementing it yourself with explicit loops, and then to learn the standard tools of a professional image-processing library (SciPy) that do the same thing efficiently.

### Visualize images
Display and get familiar with the images.

In [ ]:
# Declare image_list for the viewer
image_list = [bikesgray, camera, spots]

plt.close("all")
imgs_viewer = viewer(image_list, widgets=True, hist=True)

# 1. Edge detection

In this part of the lab, we will review one of the most basic image processing operations, edge detection. We will also use this filter to review the concepts of convolution and correlation. As you might remember from the course, edge detection algorithms try to detect sharp intensity differences between neighbouring pixels. A common vertical-edge detector (detecting changes in the $x$ direction) is the Sobel filter, represented by the impulse response

$$h[m,n] =
\begin{bmatrix}
    -1 & 0 & 1 \\
    -2 & 0 & 2 \\
    -1 & 0 & 1
\end{bmatrix}\,.$$

If the pixels surrounding a given pixel position share a similar value, then the output at that pixel will have a value close to zero after applying the mask. On the other hand, if the values of the left and right columns are significantly different, which likely indicates some kind of vertical edge, then the output at that location will have either a highly positive or highly negative intensity.

As you saw in the lecture, there are filters that you can implement in a separable way, and filters that you cannot. Since the separable way is the more efficient one, you should develop the ability to recognize filters that are separable and implement them as such. Go through the next sections to implement both methods!

## 1.1. Convolution in 1D

The building block of a separable filter is the 1D convolution of a signal $f$ (a row or a column of the image) with a 1D mask $h$:

$$(h \ast f)[x] = \sum_{i} h[i]\,f[x - i]\,.$$

Note the minus sign: the mask is *flipped* with respect to the signal. The function `filter1d` in the next cell implements this formula with an explicit loop over the positions $x$ of the signal, and an inner loop over the elements of the mask. Read it carefully: your own implementations below will follow exactly the same pattern.

**Note:** Near the borders of the signal, the mask reaches outside of it. We have to decide what values to use there: these are the *boundary conditions*. In this lab we use *mirrored* boundaries that repeat the edge sample (`... c b a | a b c ...`), which NumPy calls `mode='symmetric'` in `np.pad`, and which is the default of the SciPy filtering functions that you will use later (where it is called `mode='reflect'`). Note that the names differ between libraries: NumPy's `mode='reflect'`, SciPy's `mode='mirror'` and OpenCV's default do *not* repeat the edge sample (`... c b | a b c ...`).

In [ ]:
# 1D convolution of a signal with a 1D mask, with mirrored boundaries
def filter1d(signal, mask):
    # Extend the signal on both sides by mirroring, so that the mask never reaches outside of it
    half = len(mask) // 2
    padded = np.pad(signal, half, mode="symmetric")
    # Declare the output signal
    output = np.zeros(len(signal))
    # Iterate through all the positions of the signal
    for x in range(len(signal)):
        # Convolution: the neighbourhood padded[x : x + len(mask)] is multiplied with the flipped mask
        for i in range(len(mask)):
            output[x] += padded[x + i] * mask[len(mask) - 1 - i]
    return output


# A quick demonstration on a step signal with the central-difference mask [-1, 0, 1].
step = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0])
print("Signal:  ", step)
print("Filtered:", filter1d(step, np.array([-1, 0, 1])))

## 1.2. Separable implementation (2 points)

A separable 2D filter is applied as two 1D convolutions: every row of the image is convolved with a horizontal mask `mask_x`, and then every column of the result is convolved with a vertical mask `mask_y`. In the next cell, we provide the function `filter_separable(img, mask_x, mask_y)` with the row filtering already implemented using `filter1d`.

**For 1 point**, complete the function `filter_separable` by implementing the column filtering. The row filtering is provided and can be used as a reference: note that the columns must be taken from the `output` of the row filtering, not from the original image.

*Hint:* a column of a 2D NumPy array is accessed with `output[:, x]`, and it can be assigned in the same way.

In [ ]:
# Function that performs a separable convolution on img using the two 1D masks mask_x and mask_y
def filter_separable(img, mask_x, mask_y):
    # Declare the output image, of the same shape as the input
    output = np.zeros(img.shape)

    # Iterate through every row: extract the row, filter it with mask_x and store it in the output
    for y in range(img.shape[0]):
        output[y, :] = filter1d(img[y, :], mask_x)

    # Iterate through every column of the output: extract the column, filter it with mask_y and store it back
    # YOUR CODE HERE

    return output

### 1D masks

Before testing your separable implementation, **for 1 point**, define in the next cell the two 1D masks `mask_x` (horizontal) and `mask_y` (vertical) that correspond to the separable version of the Sobel vertical-edge detector presented at the beginning of this section. The 2D mask must be the *outer product* of the two 1D masks: $h[m, n] = \mathrm{mask\_y}[m]\cdot\mathrm{mask\_x}[n]$.

**Note:** This factorization is not unique: if you multiply one mask by a constant and divide the other by the same constant, their outer product does not change. Any pair of masks whose outer product is the Sobel mask is accepted.

In [ ]:
# Change the masks to the correct values
mask_x = np.array([0, 0, 0])
mask_y = np.array([0, 0, 0])

# YOUR CODE HERE

Now, run the next cell for a quick sanity check on your 1D masks and on your function `filter_separable`. We convolve an impulse image with your 1D masks: the result should be exactly the Sobel mask presented at the beginning of the section (a convolution with an impulse returns the impulse response, i.e. the mask itself).

In [ ]:
# Sanity check with an impulse image
impulse = np.zeros((3, 3))
impulse[1, 1] = 1
# Declare the correct 2D mask
sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
# The result of a convolution with an impulse image should be the mask itself
assert np.shape(mask_x) == (3,) and np.shape(mask_y) == (3,), (
    "mask_x and mask_y should be 1D arrays of length 3."
)
result = filter_separable(impulse, mask_x, mask_y)
print("Convolving an impulse should return the original 2D mask.\nMask:\n", sobel)
print("Result:\n", result)
if np.allclose(result, sobel, atol=1e-9):
    print("\nThe separable filter and the 1D masks have passed this sanity check!")
elif np.allclose(result, sobel, atol=1e-3):
    print(
        "\nWARNING!!!\nYour masks are almost correct, but their precision does not seem to be enough: use exact values."
    )
else:
    print(
        "\nWARNING!!!\nSorry, it looks like the separable filter or the 1D masks are not yet correct."
    )

Now run the next cell to apply your function to the three real images and to display the results.

In [ ]:
# Apply the function to the images
vedge_bike_s = filter_separable(bikesgray, mask_x, mask_y)
vedge_spots_s = filter_separable(spots, mask_x, mask_y)
vedge_camera_s = filter_separable(camera, mask_x, mask_y)

# Visualize the results
plt.close("all")
vedge_s_viewer = viewer(
    [vedge_bike_s, vedge_camera_s, vedge_spots_s],
    title=[
        "Bike - Vertical edges",
        "Camera - Vertical edges",
        "Spots - Vertical edges",
    ],
    widgets=True,
)

## 1.3. Non-separable implementation (1 point)

In this section, we will implement a 2D convolution function for filters that are not separable. The two-dimensional convolution between an image $f$ and a mask $h$ is given by

$$(h \ast f)[x,y] = \sum_{m}\sum_{n} f[x-m,\,y-n]\,h[m,n]\,,$$

i.e. the same flipping of the mask as in 1D, now in both directions. We will test it on the following non-separable mask:

$$\begin{bmatrix}
        -3  & 1 & 3 \\
        -10 & 1 & 10 \\
        -3  & 1 & 3
    \end{bmatrix}.$$

In the next cell we provide the skeleton of the function `filter_non_separable(img, mask)`: the image is already extended with mirrored boundaries (`padded`), so that the neighbourhood of the pixel `(y, x)` of the image is `padded[y : y + mask_h, x : x + mask_w]`, and the two loops over the pixels of the image are in place. **For 1 point**, complete the function: for every pixel, accumulate in `value` the products between the pixels of the neighbourhood and the elements of the mask, taken in the right order. Your function must work for any mask size, not only $3 \times 3$.

*Hint:* look at how the inner loop of `filter1d` reads the mask, and do the same along both dimensions.

In [ ]:
# Function that performs a convolution of the image img with the 2D mask, with mirrored boundaries
def filter_non_separable(img, mask):
    # Size of the mask and of its half-width
    mask_h, mask_w = mask.shape
    half_h, half_w = mask_h // 2, mask_w // 2
    # Extend the image by mirroring, so that the mask never reaches outside of it
    padded = np.pad(img, ((half_h, half_h), (half_w, half_w)), mode="symmetric")
    # Declare the output image, of the same shape as the input
    output = np.zeros(img.shape)
    # Iterate through every pixel of the image
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            # The neighbourhood of the pixel (y, x) is padded[y : y + mask_h, x : x + mask_w]
            value = 0.0
            # YOUR CODE HERE
            # Assign the value to the output image
            output[y, x] = value
    return output

Run the next cell for a sanity check: the convolution of an impulse with a mask must return the mask itself. We test with the symmetric mask above *and* with a non-symmetric mask, because an implementation that forgets to flip the mask (a *correlation* instead of a convolution) gives the right result on symmetric masks only.

In [ ]:
# Sanity check with a simple 3x3 impulse image
impulse = np.zeros((3, 3))
impulse[1, 1] = 1
# Declare the masks
mask = np.array([[-3, 1, 3], [-10, 1, 10], [-3, 1, 3]])
nonsymmetric_mask = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
# Check the results
result = filter_non_separable(impulse, mask)
result_nonsymmetric = filter_non_separable(impulse, nonsymmetric_mask)
assert np.shape(result) == (3, 3), (
    "The output should have the same shape as the input image."
)
print(
    "Convolving an impulse should return the unchanged mask.\nSymmetric mask:\n",
    mask,
    "\nResult:\n",
    result,
)
print("\nNon-symmetric mask:\n", nonsymmetric_mask, "\nResult:\n", result_nonsymmetric)
if not np.allclose(result, mask):
    print(
        "\nWARNING!!!\nThe function `filter_non_separable` still provides an incorrect output."
    )
elif not np.allclose(result_nonsymmetric, nonsymmetric_mask):
    print(
        "\nWARNING!!!\nThe function works with a symmetric mask, but not with a non-symmetric one: it computes a correlation instead of a convolution. Remember to flip the mask."
    )
else:
    print("\nThe function works with both the symmetric and the non-symmetric mask!")

### Applying the function

Run the next cell to apply `filter_non_separable` with the mask given above to the images `bikesgray`, `camera` and `spots`, and to display the results.

In [ ]:
# Apply the non-separable filter to the images
vedge_bike_nons = filter_non_separable(bikesgray, mask)
vedge_spots_nons = filter_non_separable(spots, mask)
vedge_camera_nons = filter_non_separable(camera, mask)

# Visualize the results
plt.close("all")
vedge_nons_viewer = viewer(
    [vedge_bike_nons, vedge_camera_nons, vedge_spots_nons],
    title=[
        "Bike - Vertical edges",
        "Camera - Vertical edges",
        "Spots - Vertical edges",
    ],
    widgets=True,
)

## 1.4. Comparison: separable vs non-separable

The Sobel filter is separable, so it can be applied with both of your implementations. Are the results the same? If both your implementations are correct, the difference should be negligible. Run the next cell to apply `filter_non_separable` with the 2D Sobel mask to the three images, and to compare the computation time of the two implementations.

The separable implementation needs $2 \times 3 = 6$ multiplications per pixel for a $3 \times 3$ mask, whereas the non-separable one needs $3 \times 3 = 9$: for a mask of size $N \times N$, this is $2N$ against $N^2$, which is why separable filters should always be implemented as such.

In [ ]:
# Apply the non-separable implementation with the 2D Sobel mask
sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
start = time.perf_counter()
vedge_bike_nons = filter_non_separable(bikesgray, sobel)
time_nons = time.perf_counter() - start
vedge_spots_nons = filter_non_separable(spots, sobel)
vedge_camera_nons = filter_non_separable(camera, sobel)

# Time the separable implementation on the same image
start = time.perf_counter()
filter_separable(bikesgray, mask_x, mask_y)
time_s = time.perf_counter() - start

print(
    f"Non-separable implementation: {time_nons:.2f} s, separable implementation: {time_s:.2f} s on the image bikesgray."
)
print(
    f"Maximum absolute difference between the two results: {np.max(np.abs(vedge_bike_nons - vedge_bike_s)):.2e}"
)

Now, go ahead and do some exploration! Try to plot some of the images next to each other, look at the two histograms, zoom into different areas of the three images, and see if they are truly equal. There are several tests you can do to measure the equality of both implementations. You can take the difference between the two images and visualize it (`image1 - image2`), or you can use the functions provided by NumPy to compare arrays (see [numpy.testing.assert_array_equal](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_array_equal.html) and [numpy.testing.assert_array_almost_equal](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_array_almost_equal.html)).

Use the next cell to do any exploration you want. The results of the separable implementation are in `vedge_bike_s`, `vedge_camera_s` and `vedge_spots_s`, and those of the non-separable one in `vedge_bike_nons`, `vedge_camera_nons` and `vedge_spots_nons`.

In [ ]:
# Use this cell for your exploration
# YOUR CODE HERE

## 1.5. Edge detection with SciPy

In Python, the two functions `filter_non_separable` and `filter_separable` that you implemented correspond to [`convolve`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.convolve.html) and [`convolve1d`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.convolve1d.html) from the [`scipy.ndimage`](https://docs.scipy.org/doc/scipy/reference/ndimage.html) library (imported as `ndi`), which are implemented in compiled code and are therefore much faster. We will compare your implementations with them. Read the documentation of `convolve` and `convolve1d`, and pay attention to how they are used in the next cell, as you will use them in the next section. Note that their default boundary condition, `mode='reflect'`, is the mirroring with repeated edge sample that we used in our implementations.

If you did everything correctly in sections 1.2 and 1.3, the next cell will display a winning message. Otherwise, it will plot your images with the regions that differ from the correct answer highlighted in red, which should give you a clue about where you might have gone wrong. For this purpose we use the function [`np.allclose`](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html), a standard NumPy comparison tool. Furthermore, you can uncomment the final line to visualize the four edge detections we have so far.

**Note:** In this lab we have been using `float64` ($64$-bit floating point) as the `dtype` (which stands for *data type*) of the images (look at the Imports section). This `dtype` is the most versatile, and since the `dtype` of the output array is the same (by default) as the `dtype` of the input array, we do not really need to mind the `output` parameter right now.

However, it is also very common to store images and work with them as `np.uint8` (8-bit unsigned integer, which spans the range $[0, 255]$). Nonetheless, we know that the gradient can take large positive and negative values, so an [unsigned integer](https://en.wikipedia.org/wiki/Integer_%28computer_science%29) is not the right type. If you do not realize why, do a quick experiment: declare a NumPy array of length one, of data type `'uint8'`, and assign a value outside of the range $[0, 255]$ (e.g. `np.array([300], dtype='uint8')`). Perform a few operations and see what happens.

If at some point you need to work with integers (e.g. to save memory or to increase performance), because we know that the result will still be an integer (the mask only contains integers), you can choose `np.int32` (32-bit signed integers). This is something that you will have to keep in mind constantly when working with images, and that can be the source of many bugs.

In [ ]:
# Non-separable implementation with the 2D Sobel mask
vedge_ndi = ndi.convolve(bikesgray, sobel)

# Separable implementation with the 1D masks you defined above: filter the columns, then the rows
vedge_ndi_s = ndi.convolve1d(bikesgray, mask_y, axis=0)
vedge_ndi_s = ndi.convolve1d(vedge_ndi_s, mask_x, axis=1)

# Compare with your implementations, and visualize the differences if necessary
plt.close("all")
if np.allclose(vedge_ndi, vedge_bike_nons):
    print("Your non-separable implementation is correct!")
else:
    print("Review your non-separable implementation!")
    viewer(
        [vedge_bike_nons, vedge_ndi],
        title=["Your non-separable", "SciPy non-separable"],
        compare=True,
    )

if np.allclose(vedge_ndi_s, vedge_bike_s):
    print("Congratulations!\nYour separable filter is correct!")
else:
    print(
        "Review your separable implementation!\nThe problem could be either in your 1D masks, or in the implementation itself. The previous sanity checks and the red areas should guide you."
    )
    viewer(
        [vedge_bike_s, vedge_ndi_s],
        title=["Your separable", "SciPy separable"],
        compare=True,
    )

# Uncomment the following line to visualize the four edge detections we have done so far
vedge_viewer = viewer([vedge_bike_nons, vedge_ndi, vedge_bike_s, vedge_ndi_s], title=['Your non-separable', 'SciPy non-separable', 'Your separable', 'SciPy separable'], subplots=(2, 2))

# 2. Implementation and classification of digital filters

In this exercise, we will practice the implementation and classification of masks or digital filters. Moreover, you will practice separable and non-separable implementations with SciPy. You will implement 6 functions that correspond to 6 different digital filters. Each of them takes an input image and outputs the filtered version of it. You need to
* hard-code the mask inside the function,
* apply the filtering using either `ndi.convolve` (non-separable masks) or `ndi.convolve1d` twice (separable masks), depending on whether the filter is separable.

**If a mask is separable and you do not implement it as such, you will not get the points.** Implementing each function is worth **0.5 points**, classifying its visual effect is worth **0.5 points**.

You can define 1D and 2D filters as 1D and 2D NumPy arrays, e.g.
```python
mask = np.array([1, 2])
```
```python
mask = np.array([[ 1,  2],
                 [ 6,  7]])
```

**Note:** Some filters, in particular simple edge detector filters, can intuitively be recognized as separable, and their 1D masks can also be intuitively proposed. From these 1D masks, a simple test consists in taking the outer product between them, which should result in the original filter. However, there is a more systematic approach. As you should know, any matrix $\mathbf{M}\in\mathbb{R}^{m\times n}$ can be factorized into its components $\mathbf{U}\mathbf{S}\mathbf{V}^{T}$ by Singular Value Decomposition, where $\mathbf{U}\in\mathbb{R}^{m\times m}$, $\mathbf{S}\in\mathbb{R}^{\min(m, n)\times \min(m, n)}$ is a diagonal matrix with the singular values of $\mathbf{M}$, and $\mathbf{V}\in\mathbb{R}^{n\times n}$. It follows from the definition of the matrix product that if only one singular value is larger than zero, the filter is separable. Nonetheless, for simple filters, we still recommend you to use the *intuitive* masks. If you want to extract the masks using SVD, you can use the [`svd`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html) function of the NumPy linear algebra package; we leave you the task of extracting the actual 1D masks from it.

## 2.1. Mask A (1 point)
```python
    [ 0,  0,  1,  0,  0]
    [ 0,  1,  2,  1,  0]
A = [ 1,  2,-16,  2,  1]
    [ 0,  1,  2,  1,  0]
    [ 0,  0,  1,  0,  0]
```

**For 0.5 points**, complete the function `filter_A(img)` in the next cell, which filters the image `img` with the mask A above.

In [ ]:
# Function that filters img with the mask A
def filter_A(img):
    output = img.copy()

    # Filter the image with mask A
    # YOUR CODE HERE

    return output

Run the next cell to visualize the result of applying `filter_A` to the image `bikesgray`. 

In [ ]:
# Apply the function
bikesgray_A = filter_A(bikesgray)
assert bikesgray_A is not None and np.shape(bikesgray_A) == bikesgray.shape, (
    "filter_A must return an image of the same shape as its input."
)
# Visualize the result
plt.close("all")
vis_A = viewer(
    [bikesgray, bikesgray_A],
    title=["Original", "Filtered with mask A"],
    subplots=(1, 2),
)

**For 0.5 points**, answer the following MCQ:

* Which of the following keywords best describes the visual effect of this filter?

1. isotropic blurring, or
2. vertical blurring, or
3. horizontal blurring, or
4. diagonal blurring, or
5. sharpening, or
6. vertical-edge detection, or
7. horizontal-edge detection, or
8. diagonal edge detection, or
9. Laplacian: a second order derivative to find edges (maxima and minima), or
10. it has no effect.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
# Sanity check
if not answer in list(range(1, 11)):
    print("WARNING!\nAnswer one of 1, 2, 3, 4, 5, 6, 7, 8, 9 or 10.")

## 2.2. Mask B (1 point)
```python
    [ 0,  1,  2,  1,  0]
    [ 0,  1,  2,  1,  0]
B = [ 0,  1,  2,  1,  0]
    [ 0,  1,  2,  1,  0]
    [ 0,  1,  2,  1,  0]
```

**For 0.5 points**, complete the function `filter_B(img)` in the next cell, which filters the image `img` with the mask B above.

In [ ]:
# Function that filters img with the mask B
def filter_B(img):
    output = img.copy()

    # Filter the image with mask B
    # YOUR CODE HERE

    return output

Run the next cell to visualize the result of applying `filter_B` to the image `bikesgray`.

In [ ]:
# Apply the function
bikesgray_B = filter_B(bikesgray)
assert bikesgray_B is not None and np.shape(bikesgray_B) == bikesgray.shape, (
    "filter_B must return an image of the same shape as its input."
)
# Visualize the result
plt.close("all")
vis_B = viewer(
    [bikesgray, bikesgray_B],
    title=["Original", "Filtered with mask B"],
    subplots=(1, 2),
)

**For 0.5 points**, answer the following MCQ:

* Which of the following keywords best describes the visual effect of this filter?

1. isotropic blurring, or
2. vertical blurring, or
3. horizontal blurring, or
4. diagonal blurring, or
5. sharpening, or
6. vertical-edge detection, or
7. horizontal-edge detection, or
8. diagonal edge detection, or
9. Laplacian: a second order derivative to find edges (maxima and minima), or
10. it has no effect.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
# Sanity check
if not answer in list(range(1, 11)):
    print("WARNING!\nAnswer one of 1, 2, 3, 4, 5, 6, 7, 8, 9 or 10.")

## 2.3. Mask C (1 point)
```python
    [-1,  -1,  -1,  -1,  -1]
    [-2,  -2,  -2,  -2,  -2]
C = [ 0,   0,   0,   0,   0]
    [ 2,   2,   2,   2,   2]
    [ 1,   1,   1,   1,   1]
```

**For 0.5 points**, complete the function `filter_C(img)` in the next cell, which filters the image `img` with the mask C above.

In [ ]:
# Function that filters img with the mask C
def filter_C(img):
    output = img.copy()

    # Filter the image with mask C
    # YOUR CODE HERE

    return output

Run the next cell to visualize the result of applying `filter_C` to the image `bikesgray`.

In [ ]:
# Apply the function
bikesgray_C = filter_C(bikesgray)
assert bikesgray_C is not None and np.shape(bikesgray_C) == bikesgray.shape, (
    "filter_C must return an image of the same shape as its input."
)
# Visualize the result
plt.close("all")
vis_C = viewer(
    [bikesgray, bikesgray_C],
    title=["Original", "Filtered with mask C"],
    subplots=(1, 2),
)

**For 0.5 points**, answer the following MCQ:

* Which of the following keywords best describes the visual effect of this filter?

1. isotropic blurring, or
2. vertical blurring, or
3. horizontal blurring, or
4. diagonal blurring, or
5. sharpening, or
6. vertical-edge detection, or
7. horizontal-edge detection, or
8. diagonal edge detection, or
9. Laplacian: a second order derivative to find edges (maxima and minima), or
10. it has no effect.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
# Sanity check
if not answer in list(range(1, 11)):
    print("WARNING!\nAnswer one of 1, 2, 3, 4, 5, 6, 7, 8, 9 or 10.")

## 2.4. Mask D (1 point)
```python
    [-1,  -4,  -6,  -4, -1]
    [-4, -16, -24, -16, -4]
D = [-6, -24, 476, -24, -6]
    [-4, -16, -24, -16, -4]
    [-1,  -4,  -6,  -4, -1]
```

**For 0.5 points**, complete the function `filter_D(img)` in the next cell, which filters the image `img` with the mask D above.

In [ ]:
# Function that filters img with the mask D
def filter_D(img):
    output = img.copy()

    # Filter the image with mask D
    # YOUR CODE HERE

    return output

Run the next cell to visualize the result of applying `filter_D` to the image `bikesgray`.

In [ ]:
# Apply the function
bikesgray_D = filter_D(bikesgray)
assert bikesgray_D is not None and np.shape(bikesgray_D) == bikesgray.shape, (
    "filter_D must return an image of the same shape as its input."
)
# Visualize the result
plt.close("all")
vis_D = viewer(
    [bikesgray, bikesgray_D],
    title=["Original", "Filtered with mask D"],
    subplots=(1, 2),
)

**For 0.5 points**, answer the following MCQ:

* Which of the following keywords best describes the visual effect of this filter?

1. isotropic blurring, or
2. vertical blurring, or
3. horizontal blurring, or
4. diagonal blurring, or
5. sharpening, or
6. vertical-edge detection, or
7. horizontal-edge detection, or
8. diagonal edge detection, or
9. Laplacian: a second order derivative to find edges (maxima and minima), or
10. it has no effect.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
# Sanity check
if not answer in list(range(1, 11)):
    print("WARNING!\nAnswer one of 1, 2, 3, 4, 5, 6, 7, 8, 9 or 10.")

## 2.5. Mask E (1 point)
```python
    [1, 1, 1, 1, 1]
    [1, 1, 1, 1, 1]
E = [1, 1, 1, 1, 1]
    [1, 1, 1, 1, 1]
    [1, 1, 1, 1, 1]
```

**For 0.5 points**, complete the function `filter_E(img)` in the next cell, which filters the image `img` with the mask E above.

In [ ]:
# Function that filters img with the mask E
def filter_E(img):
    output = img.copy()

    # Filter the image with mask E
    # YOUR CODE HERE

    return output

Run the next cell to visualize the result of applying `filter_E` to the image `bikesgray`.

In [ ]:
# Apply the function
bikesgray_E = filter_E(bikesgray)
assert bikesgray_E is not None and np.shape(bikesgray_E) == bikesgray.shape, (
    "filter_E must return an image of the same shape as its input."
)
# Visualize the result
plt.close("all")
vis_E = viewer(
    [bikesgray, bikesgray_E],
    title=["Original", "Filtered with mask E"],
    subplots=(1, 2),
)

**For 0.5 points**, answer the following MCQ:

* Which of the following keywords best describes the visual effect of this filter?

1. isotropic blurring, or
2. vertical blurring, or
3. horizontal blurring, or
4. diagonal blurring, or
5. sharpening, or
6. vertical-edge detection, or
7. horizontal-edge detection, or
8. diagonal edge detection, or
9. Laplacian: a second order derivative to find edges (maxima and minima), or
10. it has no effect.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
# Sanity check
if not answer in list(range(1, 11)):
    print("WARNING!\nAnswer one of 1, 2, 3, 4, 5, 6, 7, 8, 9 or 10.")

## 2.6. Mask F (1 point)
```python
    [ 0, -1, -1,  0,  0]
    [-1, -1,  0,  0,  0]
F = [-1,  0,  0,  0,  1]
    [ 0,  0,  0,  1,  1]
    [ 0,  0,  1,  1,  0]
```

**For 0.5 points**, complete the function `filter_F(img)` in the next cell, which filters the image `img` with the mask F above.

In [ ]:
# Function that filters img with the mask F
def filter_F(img):
    output = img.copy()

    # Filter the image with mask F
    # YOUR CODE HERE

    return output

Run the next cell to visualize the result of applying `filter_F` to the image `bikesgray`.

In [ ]:
# Apply the function
bikesgray_F = filter_F(bikesgray)
assert bikesgray_F is not None and np.shape(bikesgray_F) == bikesgray.shape, (
    "filter_F must return an image of the same shape as its input."
)
# Visualize the result
plt.close("all")
vis_F = viewer(
    [bikesgray, bikesgray_F],
    title=["Original", "Filtered with mask F"],
    subplots=(1, 2),
)

**For 0.5 points**, answer the following MCQ:

* Which of the following keywords best describes the visual effect of this filter?

1. isotropic blurring, or
2. vertical blurring, or
3. horizontal blurring, or
4. diagonal blurring, or
5. sharpening, or
6. vertical-edge detection, or
7. horizontal-edge detection, or
8. diagonal edge detection, or
9. Laplacian: a second order derivative to find edges (maxima and minima), or
10. it has no effect.

In [ ]:
# Assign your answer to this variable
answer = None
# YOUR CODE HERE

In [ ]:
# Sanity check
if not answer in list(range(1, 11)):
    print("WARNING!\nAnswer one of 1, 2, 3, 4, 5, 6, 7, 8, 9 or 10.")

🎉 Congratulations on finishing the first part of the Filtering lab!

Make sure to save your notebook (you might want to keep a copy on your personal computer) and upload it to Moodle, **in a zip file with the other notebook of this lab.**

* Keep the name of the notebook as: *1_filtering.ipynb*,
* Name the zip file: *filtering_lab.zip*.